# 10 梯度提升 Gradient Boosting

依赖安装说明：`pip install numpy matplotlib scikit-learn`

Gradient Boosting 也是 boosting，但它的直觉是：每一轮新模型都去拟合当前模型还没解释掉的残差或负梯度。


## 1. 数学逻辑

模型是多个弱模型相加：

$$F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta h_m(x)$$

平方误差回归里，负梯度就是残差：

$$r_i = y_i - F_{m-1}(x_i)$$

所以每一轮训练一棵小树拟合残差，然后加回原模型。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)
X = np.linspace(-3, 3, 180).reshape(-1, 1)
y = np.sin(X[:, 0]) + 0.3 * X[:, 0] + np.random.normal(scale=0.18, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [ ]:
# 从零实现平方误差下的 gradient boosting：不断拟合残差
learning_rate = 0.2
pred_train = np.full(len(y_train), y_train.mean())
pred_test = np.full(len(y_test), y_train.mean())
trees = []

for m in range(20):
    residual = y_train - pred_train
    tree = DecisionTreeRegressor(max_depth=2, random_state=m)
    tree.fit(X_train, residual)
    pred_train += learning_rate * tree.predict(X_train)
    pred_test += learning_rate * tree.predict(X_test)
    trees.append(tree)
    if m in [0, 1, 4, 9, 19]:
        print(f'round {m+1:2d} | test MSE={mean_squared_error(y_test, pred_test):.4f}')


In [ ]:
model = GradientBoostingRegressor(n_estimators=80, learning_rate=0.08, max_depth=2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('GradientBoostingRegressor MSE:', round(mean_squared_error(y_test, pred), 4))

line_x = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.scatter(X_train[:,0], y_train, s=20, alpha=0.6, label='train')
plt.plot(line_x[:,0], model.predict(line_x), color='red', label='boosting fit')
plt.legend()
plt.title('Gradient Boosting 拟合非线性函数')
plt.show()


## 2. 常见误区

- Gradient Boosting 很强，但超参数敏感，容易过拟合。
- `learning_rate` 小通常要配更多树。
- 树太深时，每轮弱学习器不再“弱”，可能过拟合。

## 3. 小实验

- 改 `learning_rate` 和 `n_estimators` 的组合。
- 改 `max_depth`，观察曲线是否变得锯齿化。
- 把噪声调大，观察过拟合。
